# Autonomous SI$^3$R calibration and control scenarios

This notebook is independent of `Main_Forecast_SIKR.ipynb`. It uses all available weekly Australian subtype counts (2008-W01--2022-W52) to estimate the 21 constant transmission rates $\beta_i^k$ and the shared dispersal coefficient $d_0$. All remaining dynamical parameters are fixed at the values reported in Tables 1 and 2. The constrained relations are $d_S=d_R=d_0$ and $d_I^k=q_kd_0$. Estimation uses conditional one-week nonlinear least squares: the preceding observation reconstructs the infectious state, the ODE advances that state for one week, and its integrated incidence is compared with the next observation. This avoids treating 15 years of recurrent seasonal introductions as one uninterrupted autonomous outbreak. After calibration, the autonomous model is projected for 26 weeks under no control and under three single-factor controls.


In [ ]:
from pathlib import Path
from datetime import date
from time import perf_counter
import warnings
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import least_squares
from IPython.display import display

def resolve_project_dir(required_filename):
    """Find a co-located input workbook without machine-specific paths."""
    override = os.environ.get("SIKR_PROJECT_DIR")
    if override:
        project_dir = Path(override).expanduser().resolve()
        if not (project_dir / required_filename).is_file():
            raise FileNotFoundError(
                f"SIKR_PROJECT_DIR does not contain {required_filename}: {project_dir}"
            )
        return project_dir

    candidates = []
    if globals().get("__file__"):
        candidates.append(Path(__file__).resolve().parent)
    working_dir = Path.cwd().resolve()
    candidates.extend([working_dir, *working_dir.parents])
    for candidate in dict.fromkeys(candidates):
        if (candidate / required_filename).is_file():
            return candidate
    raise FileNotFoundError(
        f"Cannot locate {required_filename}. Keep the workbook beside this notebook "
        "and start Jupyter from that folder, or set SIKR_PROJECT_DIR to the folder."
    )


PROJECT_DIR = resolve_project_dir("3DataAustraliaNoAgeNoSex.xlsx")
DATA_PATH = PROJECT_DIR / "3DataAustraliaNoAgeNoSex.xlsx"
OUTPUT_DIR = PROJECT_DIR / "control_sikr_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATES = ["NSW", "Vic", "QLD", "SA", "WA", "TAS", "NT"]
SHEET_MAP = {"NSW":"NSW", "Vic":"Vic", "QLD":"Qld", "SA":"SA", "WA":"WA", "TAS":"Tas", "NT":"NT"}
STRAINS = ["H3N2", "H1N1", "B"]
STRAIN_MAP = {"H3N2":"A(H3N2)", "H1N1":"A(H1N1)", "B":"B"}
M, K = len(STATES), len(STRAINS)

# Fixed values from Tables 1 and 2, in STATES/STRAINS order.
B_FIXED = np.array([1856.40,1492.85,1193.58,379.47,671.70,111.50,75.95], dtype=float)
MU_FIXED = np.array([1.31374e-4,1.22434e-4,1.18322e-4,1.51732e-4,1.07574e-4,1.69783e-4,8.72950e-5], dtype=float)
OMEGA_FIXED = np.full(M, 6.9237e-3, dtype=float)
ALPHA_FIXED = np.array([6.96e-4,2.70e-4,4.64e-4], dtype=float)
GAMMA_FIXED = np.array([1.728,1.720,1.720], dtype=float)
Q_FIXED = np.array([0.252,0.269,0.261], dtype=float)
POPULATION = np.array([8153584,6613727,5322058,1820530,2785312,571517,250635], dtype=float)

# State-capital coordinates, in STATES order. They are used only to construct
# the static gravity prior, not as fitted covariates.
LATITUDE = np.array([-33.8688,-37.8136,-27.4698,-34.9285,-31.9505,-42.8821,-12.4634])
LONGITUDE = np.array([151.2093,144.9631,153.0251,138.6007,115.8605,147.3272,130.8456])
GRAVITY_THRESHOLD_FRACTION = 0.02

# Calibration settings. Bounds are wider than the former beta initialization interval
# so boundary hits can be diagnosed rather than silently imposed.
BETA_BOUNDS = (1.0e-4, 5.0)
D0_BOUNDS = (0.0, 5.0)
N_STARTS = 6
MAX_NFEV = 220
ODE_SUBSTEPS = 4
RANDOM_SEED = 20260916
FORECAST_WEEKS = 26

U_BETA_GRID = [0.10,0.20,0.30,0.40]
U_GAMMA_GRID = [0.10,0.20,0.40,0.80]  # week^-1
U_D_GRID = [0.25,0.50,0.75,1.00]

print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)


## 1. Weekly case tensor

ACT is excluded because its subtype counts are missing. Missing region--week--strain combinations among the seven retained regions are completed with zero, matching the preprocessing rule in the forecasting notebook.

In [ ]:
def iso_week_to_monday(year_week):
    s = str(year_week).replace(".0", "").zfill(6)
    return pd.Timestamp(date.fromisocalendar(int(s[:4]), int(s[4:]), 1))


def load_cases(path):
    raw, all_dates, audit = {}, set(), []
    for state in STATES:
        df = pd.read_excel(path, sheet_name=SHEET_MAP[state])
        required = {"year_week", "infection_count", "strain_type"}
        if not required.issubset(df.columns):
            raise ValueError(f"Missing columns in {state}: {required.difference(df.columns)}")
        df = df[["year_week","infection_count","strain_type"]].copy()
        df["date"] = df["year_week"].map(iso_week_to_monday)
        if df.duplicated(["date","strain_type"]).any():
            raise ValueError(f"Duplicate week-strain rows in {state}")
        raw[state] = df
        all_dates.update(df["date"])
        audit.append({"state":state,"raw_rows":len(df),"observed_weeks":df["date"].nunique()})
    dates = pd.date_range(min(all_dates), max(all_dates), freq="W-MON")
    cases = np.zeros((len(dates),M,K), dtype=float)
    for i,state in enumerate(STATES):
        pivot = (raw[state].pivot(index="date",columns="strain_type",values="infection_count")
                 .reindex(index=dates,columns=[STRAIN_MAP[k] for k in STRAINS])
                 .fillna(0.0))
        cases[:,i,:] = pivot.to_numpy(dtype=float)
    if not np.isfinite(cases).all() or (cases < 0).any():
        raise ValueError("Case tensor contains invalid values")
    return dates,cases,pd.DataFrame(audit)


DATES, CASES, DATA_AUDIT = load_cases(DATA_PATH)
print("Weeks:", len(DATES), DATES[0].date(), "to", DATES[-1].date())
print("Case tensor:", CASES.shape, "total cases:", int(CASES.sum()))
display(DATA_AUDIT)


## 2. Thresholded gravity network, states, and observation operator

For $i\ne j$, the raw gravity weight is $G_{ij}=N_iN_j/r_{ij}^2$, where $r_{ij}$ is the great-circle distance between the corresponding state capitals. An undirected edge is retained if $G_{ij}\ge 0.02\max_{a\ne b}G_{ab}$. To prevent a geographically remote region from being disconnected by this global cutoff, the strongest gravity edge incident on every region is retained as a connectivity backbone. The retained raw weights are then symmetrically normalized as $W=D_G^{-1/2}\widetilde G D_G^{-1/2}$, where $D_G=\operatorname{diag}(\widetilde G\mathbf 1)$, and the dispersal matrix is $L=W-\operatorname{diag}(W\mathbf1)$. Thus $L$ is symmetric, has zero row sums, and is no longer a complete-graph matrix. The fitted scale $d_0$ is specific to this normalization.

The initial infectious prevalence is approximated from the first weekly incidence by $I_i^k(0)=Y_{i,0}^k/(\mu_i+\alpha^k+\gamma^k)$, with $R_i(0)=0$ and $S_i(0)=N_i-\sum_kI_i^k(0)$.


In [ ]:
def haversine_km(lat, lon):
    phi, lam = np.radians(lat), np.radians(lon)
    dphi = phi[:,None] - phi[None,:]
    dlam = lam[:,None] - lam[None,:]
    a = np.sin(dphi/2.0)**2 + np.cos(phi)[:,None]*np.cos(phi)[None,:]*np.sin(dlam/2.0)**2
    return 6371.0 * 2.0 * np.arcsin(np.minimum(1.0,np.sqrt(a)))


DISTANCE_KM = haversine_km(LATITUDE, LONGITUDE)
RAW_GRAVITY = (POPULATION[:,None]*POPULATION[None,:]) / np.maximum(DISTANCE_KM,1.0)**2
np.fill_diagonal(RAW_GRAVITY,0.0)
GRAVITY_THRESHOLD = GRAVITY_THRESHOLD_FRACTION * RAW_GRAVITY.max()
RETAINED_GRAVITY = np.where(RAW_GRAVITY >= GRAVITY_THRESHOLD, RAW_GRAVITY, 0.0)

# The threshold can isolate a remote node. Add its strongest available link;
# symmetry is restored after all backbone links have been inserted.
for i in range(M):
    j = np.argmax(RAW_GRAVITY[i])
    RETAINED_GRAVITY[i,j] = RAW_GRAVITY[i,j]
    RETAINED_GRAVITY[j,i] = RAW_GRAVITY[j,i]
RETAINED_GRAVITY = np.maximum(RETAINED_GRAVITY, RETAINED_GRAVITY.T)
np.fill_diagonal(RETAINED_GRAVITY,0.0)

gravity_degree = RETAINED_GRAVITY.sum(axis=1)
if (gravity_degree <= 0).any():
    raise ValueError("Thresholded gravity graph contains an isolated region")
W = RETAINED_GRAVITY / np.sqrt(np.outer(gravity_degree,gravity_degree))
L = W - np.diag(W.sum(axis=1))
assert np.allclose(W,W.T) and np.allclose(L.sum(axis=1),0.0)

REMOVAL = MU_FIXED[:,None] + ALPHA_FIXED[None,:] + GAMMA_FIXED[None,:]
I0 = CASES[0] / REMOVAL
R0 = np.zeros(M,dtype=float)
S0 = POPULATION - I0.sum(axis=1) - R0
if (S0 <= 0).any():
    raise ValueError("Invalid initial susceptible population")
X0 = np.concatenate([S0[:,None],I0,R0[:,None]],axis=1)

network_rows=[]
for i,origin in enumerate(STATES):
    for j,destination in enumerate(STATES):
        if i != j:
            network_rows.append({"origin":origin,"destination":destination,
                                 "distance_km":DISTANCE_KM[i,j],"raw_gravity":RAW_GRAVITY[i,j],
                                 "edge_retained":bool(RETAINED_GRAVITY[i,j]>0),"normalized_weight":W[i,j]})
NETWORK_TABLE = pd.DataFrame(network_rows)
NETWORK_TABLE.to_csv(OUTPUT_DIR/"gravity_network.csv",index=False,encoding="utf-8-sig")
print(f"Gravity threshold = {GRAVITY_THRESHOLD:,.3f} ({GRAVITY_THRESHOLD_FRACTION:.0%} of maximum); retained undirected edges = {int(np.count_nonzero(np.triu(RETAINED_GRAVITY,1)))}")
display(pd.DataFrame(W,index=STATES,columns=STATES))
display(pd.DataFrame({"state":STATES,"population":POPULATION,"S0":S0,"I0_total":I0.sum(axis=1),"R0":R0}))


## 3. Autonomous SI$^3$R solver

The solver accumulates the incidence flow during every RK4 substep. Controls are included in the same right-hand side but are set to zero during calibration. Treatment transfers individuals from $I_i^k$ to $R_i$, so $u_\gamma$ appears in both equations.


In [ ]:
def rhs_and_incidence(X,beta,d0,u_beta=0.0,u_gamma=0.0,u_d=0.0):
    S = X[...,0]
    I = X[...,1:1+K]
    R = X[...,-1]
    N = np.maximum(X.sum(axis=-1),1.0)
    beta_c = (1.0-u_beta)*beta
    d_c = (1.0-u_d)*d0
    incidence = (S/N)[...,None]*beta_c*I
    lap_S = np.einsum("ij,...j->...i",L,S)
    lap_I = np.einsum("ij,...jk->...ik",L,I)
    lap_R = np.einsum("ij,...j->...i",L,R)
    dS = B_FIXED-MU_FIXED*S-incidence.sum(axis=-1)+OMEGA_FIXED*R+d_c*lap_S
    removal = MU_FIXED[:,None]+ALPHA_FIXED[None,:]+GAMMA_FIXED[None,:]+u_gamma
    dI = incidence-removal*I+d_c*Q_FIXED[None,:]*lap_I
    dR = ((GAMMA_FIXED[None,:]+u_gamma)*I).sum(axis=-1)-(MU_FIXED+OMEGA_FIXED)*R+d_c*lap_R
    return np.concatenate([dS[...,None],dI,dR[...,None]],axis=-1),incidence


def rk4_week(X,beta,d0,u_beta=0.0,u_gamma=0.0,u_d=0.0,substeps=ODE_SUBSTEPS):
    dt = 1.0/substeps
    total_inc = np.zeros_like(X[...,1:1+K],dtype=float)
    for _ in range(substeps):
        k1,j1 = rhs_and_incidence(X,beta,d0,u_beta,u_gamma,u_d)
        k2,j2 = rhs_and_incidence(np.maximum(X+0.5*dt*k1,0.0),beta,d0,u_beta,u_gamma,u_d)
        k3,j3 = rhs_and_incidence(np.maximum(X+0.5*dt*k2,0.0),beta,d0,u_beta,u_gamma,u_d)
        k4,j4 = rhs_and_incidence(np.maximum(X+dt*k3,0.0),beta,d0,u_beta,u_gamma,u_d)
        X = np.maximum(X+dt*(k1+2*k2+2*k3+k4)/6.0,0.0)
        total_inc += dt*(j1+2*j2+2*j3+j4)/6.0
        if not np.isfinite(X).all() or X.max() > 100*POPULATION.max():
            raise FloatingPointError("ODE trajectory diverged")
    return X,np.maximum(total_inc,0.0)


def simulate(beta,d0,n_weeks,X_start=X0,u_beta=0.0,u_gamma=0.0,u_d=0.0):
    X = np.array(X_start,dtype=float,copy=True)
    states = np.empty((n_weeks+1,M,K+2),dtype=float)
    weekly_incidence = np.empty((n_weeks,M,K),dtype=float)
    states[0] = X
    for t in range(n_weeks):
        X,inc = rk4_week(X,beta,d0,u_beta,u_gamma,u_d)
        states[t+1],weekly_incidence[t] = X,inc
    return states,weekly_incidence


## 4. Conditional one-week constrained nonlinear least squares

For target week $t$, the state at its beginning is reconstructed from week $t-1$ as $I_i^k=Y_{i,t-1}^k/(\mu_i+\alpha^k+\gamma^k)$, $R_i=0$, and $S_i=N_i-\sum_kI_i^k$. A vectorized one-week RK4 map produces $\widehat Y_{i,t}^k$. The residual is computed on the $\log(1+y)$ scale and divided by the empirical standard deviation of each region--strain series. This conditional nonlinear-least-squares criterion uses every observed transition while preventing a few high-count regions from determining every parameter. Six deterministic multi-start fits are compared; the lowest residual sum of squares is retained.


In [ ]:
TARGET_CASES = CASES[1:]
I_CONDITIONED = CASES[:-1]/REMOVAL[None,:,:]
R_CONDITIONED = np.zeros((len(DATES)-1,M),dtype=float)
S_CONDITIONED = POPULATION[None,:]-I_CONDITIONED.sum(axis=-1)-R_CONDITIONED
X_CONDITIONED = np.concatenate([S_CONDITIONED[...,None],I_CONDITIONED,R_CONDITIONED[...,None]],axis=-1)
LOG_SCALE = np.maximum(np.std(np.log1p(TARGET_CASES),axis=0),0.25)


def unpack_theta(theta):
    return theta[:-1].reshape(M,K),float(theta[-1])


def residual_vector(theta):
    beta,d0 = unpack_theta(theta)
    try:
        _,pred = rk4_week(X_CONDITIONED,beta,d0)
        residual = (np.log1p(pred)-np.log1p(TARGET_CASES))/LOG_SCALE[None,:,:]
        if not np.isfinite(residual).all():
            raise FloatingPointError
        return residual.ravel()
    except FloatingPointError:
        return np.full(TARGET_CASES.size,1.0e3,dtype=float)


growth = np.diff(np.log(CASES+0.5),axis=0)
growth0 = np.clip(np.nanmedian(growth,axis=0),-0.35,0.35)
beta_seed = np.clip(REMOVAL+growth0,BETA_BOUNDS[0]*10,BETA_BOUNDS[1]*0.8)
x_seed = np.r_[beta_seed.ravel(),0.02]
lower = np.r_[np.full(M*K,BETA_BOUNDS[0]),D0_BOUNDS[0]]
upper = np.r_[np.full(M*K,BETA_BOUNDS[1]),D0_BOUNDS[1]]

rng = np.random.default_rng(RANDOM_SEED)
starts = [x_seed]
for _ in range(N_STARTS-1):
    b = np.clip(beta_seed*np.exp(rng.normal(0,0.22,size=(M,K))),lower[0],upper[0])
    d = rng.uniform(0.002,0.30)
    starts.append(np.r_[b.ravel(),d])

fit_rows,fit_objects = [],[]
fit_started = perf_counter()
for run,x0 in enumerate(starts,1):
    started = perf_counter()
    result = least_squares(residual_vector,x0,bounds=(lower,upper),method="trf",
                           max_nfev=MAX_NFEV,ftol=1e-7,xtol=1e-7,gtol=1e-7,
                           x_scale="jac",verbose=0)
    elapsed = perf_counter()-started
    fit_objects.append(result)
    fit_rows.append({"start":run,"cost":float(result.cost),"optimality":float(result.optimality),
                     "nfev":int(result.nfev),"status":int(result.status),"success":bool(result.success),
                     "elapsed_seconds":elapsed,"message":str(result.message)})
    print(f"start {run}/{N_STARTS}: cost={result.cost:.6g}, nfev={result.nfev}, success={result.success}, seconds={elapsed:.1f}")

BEST_INDEX = int(np.argmin([r.cost for r in fit_objects]))
FIT_RESULT = fit_objects[BEST_INDEX]
BETA_HAT,D0_HAT = unpack_theta(FIT_RESULT.x)
FIT_END_STATES,FIT_PRED = rk4_week(X_CONDITIONED,BETA_HAT,D0_HAT)
FIT_PRED_FULL = np.concatenate([CASES[:1],FIT_PRED],axis=0)
FIT_RUNTIME_SECONDS = perf_counter()-fit_started
FIT_START_SUMMARY = pd.DataFrame(fit_rows)
print("Selected start:",BEST_INDEX+1,"total fitting seconds:",round(FIT_RUNTIME_SECONDS,2))
display(FIT_START_SUMMARY)


## 5. Parameter and fit diagnostics

The Jacobian-based intervals below are local approximations for the transformed least-squares objective.

In [ ]:
def metrics(y,p):
    y,p = np.asarray(y,float).ravel(),np.asarray(p,float).ravel()
    e = p-y
    return {"RMSE":float(np.sqrt(np.mean(e**2))),"MAE":float(np.mean(np.abs(e))),
            "NRMSE_range":float(np.sqrt(np.mean(e**2))/(np.ptp(y)+1e-8)),
            "SMAPE_percent":float(100*np.mean(2*np.abs(e)/(np.abs(y)+np.abs(p)+1e-8))),
            "ND":float(np.abs(e).sum()/(np.abs(y).sum()+1e-8)),
            "R2":float(1-(e**2).sum()/(((y-y.mean())**2).sum()+1e-8)),
            "PCC":float(np.corrcoef(y,p)[0,1]) if np.std(y)>0 and np.std(p)>0 else np.nan}

dof = max(1,FIT_RESULT.fun.size-FIT_RESULT.x.size)
sigma2 = 2*FIT_RESULT.cost/dof
jtj = FIT_RESULT.jac.T@FIT_RESULT.jac
cov = sigma2*np.linalg.pinv(jtj,rcond=1e-10)
se = np.sqrt(np.maximum(np.diag(cov),0.0))
condition_number = float(np.linalg.cond(jtj))

param_rows=[]
for i,state in enumerate(STATES):
    for k,strain in enumerate(STRAINS):
        idx=i*K+k; est=BETA_HAT[i,k]; serr=se[idx]
        param_rows.append({"parameter":f"beta_{state}_{strain}","region":state,"strain":strain,
                           "estimate":est,"std_error_local":serr,"ci95_low":max(BETA_BOUNDS[0],est-1.96*serr),
                           "ci95_high":min(BETA_BOUNDS[1],est+1.96*serr),
                           "lower_bound":BETA_BOUNDS[0],"upper_bound":BETA_BOUNDS[1],
                           "at_bound":bool(est<=BETA_BOUNDS[0]+1e-5 or est>=BETA_BOUNDS[1]-1e-5)})
param_rows.append({"parameter":"d0","region":"all","strain":"all","estimate":D0_HAT,
                   "std_error_local":se[-1],"ci95_low":max(D0_BOUNDS[0],D0_HAT-1.96*se[-1]),
                   "ci95_high":min(D0_BOUNDS[1],D0_HAT+1.96*se[-1]),
                   "lower_bound":D0_BOUNDS[0],"upper_bound":D0_BOUNDS[1],
                   "at_bound":bool(D0_HAT<=D0_BOUNDS[0]+1e-5 or D0_HAT>=D0_BOUNDS[1]-1e-5)})
PARAMETER_ESTIMATES=pd.DataFrame(param_rows)

metric_rows=[{"scope":"overall",**metrics(TARGET_CASES,FIT_PRED)}]
for k,strain in enumerate(STRAINS): metric_rows.append({"scope":strain,**metrics(TARGET_CASES[:,:,k],FIT_PRED[:,:,k])})
for i,state in enumerate(STATES): metric_rows.append({"scope":state,**metrics(TARGET_CASES[:,i,:],FIT_PRED[:,i,:])})
FIT_METRICS=pd.DataFrame(metric_rows)

print("d0 =",D0_HAT,"Jacobian condition number =",condition_number)
display(PARAMETER_ESTIMATES)
display(FIT_METRICS)


## 6. Control scenarios and 26-week autonomous projection

The last fitted state is shared by every scenario. The baseline has no control. Transmission reduction uses $(1-u_\beta)\beta_i^k$; treatment uses $\gamma^k+u_\gamma$; movement restriction uses $(1-u_d)d_0$ and therefore scales susceptible, infectious, and recovered dispersal consistently.


In [ ]:
X_LAST = FIT_END_STATES[-1]
BASE_STATES,BASE_INC = simulate(BETA_HAT,D0_HAT,FORECAST_WEEKS,X_LAST)

scenario_specs=[("baseline",0.0,0.0,0.0)]
scenario_specs += [("transmission",u,0.0,0.0) for u in U_BETA_GRID]
scenario_specs += [("treatment",0.0,u,0.0) for u in U_GAMMA_GRID]
scenario_specs += [("mobility",0.0,0.0,u) for u in U_D_GRID]

scenario_data={}
baseline_cumulative=BASE_INC.sum(axis=(0,1))
baseline_total=float(BASE_INC.sum())
summary=[]
for control,u_beta,u_gamma,u_d in scenario_specs:
    states,inc=simulate(BETA_HAT,D0_HAT,FORECAST_WEEKS,X_LAST,u_beta,u_gamma,u_d)
    scenario_data[(control,u_beta,u_gamma,u_d)]={"states":states,"incidence":inc}
    by_strain=inc.sum(axis=(0,1)); total=float(by_strain.sum())
    row={"control":control,"u_beta":u_beta,"u_gamma_per_week":u_gamma,"u_d":u_d,
         "cumulative_all":total,"effect_all_percent":100*(baseline_total-total)/baseline_total}
    for k,strain in enumerate(STRAINS):
        row[f"cumulative_{strain}"]=float(by_strain[k])
        row[f"effect_{strain}_percent"]=100*(baseline_cumulative[k]-by_strain[k])/baseline_cumulative[k]
    summary.append(row)
CONTROL_SUMMARY=pd.DataFrame(summary)
display(CONTROL_SUMMARY)


## 7. Three-by-four control figure

Columns show all infections and the three retained categories. Rows compare the no-control baseline with the four predeclared intensities of each single control over 26 weeks.


In [ ]:
history_dates = pd.DatetimeIndex(DATES)
future_dates = pd.date_range(DATES[-1]+pd.Timedelta(days=7),periods=FORECAST_WEEKS,freq="W-MON")
forecast_week = np.arange(FORECAST_WEEKS+1)
observed_nat = CASES.sum(axis=1)
fitted_nat = FIT_PRED_FULL.sum(axis=1)

def select_outcome(arr,col):
    # arr [time,strain]
    return arr.sum(axis=1) if col==0 else arr[:,col-1]

fig,axes=plt.subplots(3,4,figsize=(18,10),sharex=False)
col_titles=["All infections"]+STRAINS
row_titles=["Transmission reduction","Antiviral treatment","Mobility restriction"]
palette=["#E69F00","#009E73","#D55E00","#CC79A7"]
control_rows=[("transmission",U_BETA_GRID,r"$u_\beta$"),("treatment",U_GAMMA_GRID,r"$u_\gamma$"),("mobility",U_D_GRID,r"$u_d$")]
for c,title in enumerate(col_titles):
    base_future=np.r_[0.0,np.cumsum(select_outcome(BASE_INC.sum(axis=1),c))]
    for r in range(3):
        ax=axes[r,c]
        ax.plot(forecast_week,base_future,color="black",lw=1.8,label="No control")
        control,grid,symbol=control_rows[r]
        for color,value in zip(palette,grid):
            if control=="transmission": key=(control,value,0.0,0.0)
            elif control=="treatment": key=(control,0.0,value,0.0)
            else: key=(control,0.0,0.0,value)
            inc=scenario_data[key]["incidence"].sum(axis=1)
            future=np.r_[0.0,np.cumsum(select_outcome(inc,c))]
            ax.plot(forecast_week,future,color=color,lw=1.5,label=f"{symbol}={value:g}")
        ax.set_xlim(0,FORECAST_WEEKS)
        if r==0: ax.set_title(title,fontsize=12,fontweight="bold")
        if c==0:
            ax.set_ylabel(row_titles[r]+"\n26-week cumulative incidence")
        ax.grid(axis="y",alpha=0.25)
        ax.spines[["top","right"]].set_visible(False)
        if c==3: ax.legend(frameon=False,fontsize=8,loc="upper left")
for row in axes:
    for ax in row: ax.set_xlabel("Weeks after intervention")
fig.suptitle("Cumulative influenza incidence under single control measures: thresholded gravity network",fontsize=15,fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.975])
FIG_PNG=OUTPUT_DIR/"control_measures_3x4_gravity.png"
FIG_PDF=OUTPUT_DIR/"control_measures_3x4_gravity.pdf"
fig.savefig(FIG_PNG,dpi=240,bbox_inches="tight",facecolor="white")
fig.savefig(FIG_PDF,bbox_inches="tight",facecolor="white")
plt.show()


## 8. Region-specific mobility-restriction trajectories

Each panel aggregates the three strains within one region. It compares the 26-week cumulative incidence under no mobility restriction with the four prespecified movement-restriction intensities.


In [ ]:
fig,axes=plt.subplots(2,4,figsize=(18,8),sharex=True)
axes=axes.ravel()
mobility_palette=["#E69F00","#009E73","#D55E00","#CC79A7"]
baseline_by_region=np.r_[np.zeros((1,M)),np.cumsum(BASE_INC.sum(axis=2),axis=0)]
for i,(state,ax) in enumerate(zip(STATES,axes)):
    ax.plot(forecast_week,baseline_by_region[:,i],color="black",lw=2.0,label="No control")
    for color,u_d in zip(mobility_palette,U_D_GRID):
        inc=scenario_data[("mobility",0.0,0.0,u_d)]["incidence"]
        cumulative=np.r_[0.0,np.cumsum(inc[:,i,:].sum(axis=1))]
        ax.plot(forecast_week,cumulative,color=color,lw=1.7,label=rf"$u_d={u_d:g}$")
    ax.set_title(state,fontweight="bold")
    ax.set_xlim(0,FORECAST_WEEKS)
    ax.set_xlabel("Weeks after intervention")
    ax.set_ylabel("Cumulative incidence")
    ax.grid(axis="y",alpha=0.25)
    ax.spines[["top","right"]].set_visible(False)
    ax.legend(frameon=False,fontsize=8,loc="upper left")
axes[-1].axis("off")
fig.suptitle("Region-specific cumulative incidence under mobility restriction",fontsize=15,fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.95])
REGION_MOBILITY_PNG=OUTPUT_DIR/"mobility_restriction_by_region.png"
REGION_MOBILITY_PDF=OUTPUT_DIR/"mobility_restriction_by_region.pdf"
fig.savefig(REGION_MOBILITY_PNG,dpi=240,bbox_inches="tight",facecolor="white")
fig.savefig(REGION_MOBILITY_PDF,bbox_inches="tight",facecolor="white")
plt.show()


## 9. Reproducible exports

The weekly tables retain region- and strain-specific trajectories. `fit_weekly_predictions.csv` covers all observed weeks; `control_weekly_trajectories.csv` covers the 26-week intervention period for every scenario.


In [ ]:
PARAMETER_ESTIMATES.to_csv(OUTPUT_DIR/"estimated_parameters.csv",index=False,encoding="utf-8-sig")
FIT_START_SUMMARY.to_csv(OUTPUT_DIR/"multistart_fit_summary.csv",index=False,encoding="utf-8-sig")
FIT_METRICS.to_csv(OUTPUT_DIR/"fit_metrics.csv",index=False,encoding="utf-8-sig")
CONTROL_SUMMARY.to_csv(OUTPUT_DIR/"control_scenario_summary.csv",index=False,encoding="utf-8-sig")

fit_rows=[]
for t,dt in enumerate(DATES):
    for i,state in enumerate(STATES):
        for k,strain in enumerate(STRAINS):
            fit_rows.append({"date":dt.strftime("%Y-%m-%d"),"year_week":f"{dt.isocalendar().year}{dt.isocalendar().week:02d}",
                             "region":state,"strain":strain,"observed":CASES[t,i,k],"fitted":FIT_PRED_FULL[t,i,k],
                             "residual":FIT_PRED_FULL[t,i,k]-CASES[t,i,k],
                             "fit_role":"initialization" if t==0 else "conditional_one_week"})
pd.DataFrame(fit_rows).to_csv(OUTPUT_DIR/"fit_weekly_predictions.csv",index=False,encoding="utf-8-sig")

control_rows=[]
for (control,u_beta,u_gamma,u_d),values in scenario_data.items():
    inc=values["incidence"]
    for t,dt in enumerate(future_dates):
        for i,state in enumerate(STATES):
            for k,strain in enumerate(STRAINS):
                control_rows.append({"date":dt.strftime("%Y-%m-%d"),"forecast_week":t+1,"control":control,
                                     "u_beta":u_beta,"u_gamma_per_week":u_gamma,"u_d":u_d,
                                     "region":state,"strain":strain,"weekly_incidence":inc[t,i,k]})
pd.DataFrame(control_rows).to_csv(OUTPUT_DIR/"control_weekly_trajectories.csv",index=False,encoding="utf-8-sig")

pd.DataFrame({"quantity":["fit_runtime_seconds","jacobian_condition_number","selected_start","n_observed_weeks"],
              "value":[FIT_RUNTIME_SECONDS,condition_number,BEST_INDEX+1,len(DATES)]}).to_csv(
              OUTPUT_DIR/"calibration_diagnostics.csv",index=False,encoding="utf-8-sig")
np.savez_compressed(OUTPUT_DIR/"calibration_and_control_arrays.npz",dates=np.array(DATES.strftime("%Y-%m-%d")),
                    cases=CASES,fitted=FIT_PRED_FULL,conditioned_start_states=X_CONDITIONED,
                    conditioned_end_states=FIT_END_STATES,beta=BETA_HAT,d0=D0_HAT,L=L,
                    W=W,distance_km=DISTANCE_KM,raw_gravity=RAW_GRAVITY,
                    retained_gravity=RETAINED_GRAVITY,population=POPULATION)
print("Saved outputs to",OUTPUT_DIR)
